# Notebook para Processamento do PDF e Geração do FAQ
Este notebook:
- Lê o PDF
- Extrai Perguntas e Respostas usando GPT-4 via API OpenAI
- Gera um arquivo `faq.json`
- Permite exportar o arquivo JSON para download


In [ ]:
# Instalação de bibliotecas necessárias
!pip install openai PyPDF2

In [ ]:
# Configuração da API Key
import openai

openai.api_key = 'SUA_OPENAI_API_KEY_AQUI'  # Substitua pela sua chave

In [ ]:
# Função para ler o PDF
import PyPDF2

def ler_pdf(caminho_pdf):
    with open(caminho_pdf, 'rb') as f:
        leitor = PyPDF2.PdfReader(f)
        texto = ""
        for pagina in leitor.pages:
            texto += pagina.extract_text()
    return texto

pdf_path = '/content/500perguntasabacaxi.pdf'  # Ajuste o caminho se necessário
texto_pdf = ler_pdf(pdf_path)
print(texto_pdf[:1000])  # Visualizar os primeiros caracteres

In [ ]:
# Função para extrair perguntas e respostas com GPT-4
def extrair_faq(texto):
    prompt = f"""
    Extraia todas as perguntas e respostas deste texto. Formate como uma lista JSON no formato:
    [
      {{"pergunta": "...", "resposta": "..."}},
      ...
    ]
    Texto:
    '''{texto}'''
    """
    resposta = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return resposta['choices'][0]['message']['content']

faq_extraido = extrair_faq(texto_pdf)
print(faq_extraido[:1000])  # Visualiza parte do resultado bruto

In [ ]:
# Salvando o FAQ extraído como JSON
faq_json = json.loads(faq_extraido)
with open('faq.json', 'w', encoding='utf-8') as f:
    json.dump(faq_json, f, ensure_ascii=False, indent=4)

print("FAQ salvo como faq.json")

In [ ]:
# Gerar link para download do faq.json no Colab
from google.colab import files
files.download('faq.json')